# Kaggle - Brain Tumor MRI Dataset

You can find the dataset and some informations about on the [Kaggle page](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

For details on steps below, please see documentation in the *docs* directory.

## Google Collab setup
### Installations

In [1]:
!pip install mlflow

### Data location

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
os.listdir('/content/drive/MyDrive/data/Brain Tumor MRI')

['data', 'models']

## General

In [4]:
import mlflow
mlflow.set_tracking_uri("https://preachingly-nonabjuratory-marget.ngrok-free.dev")
mlflow.set_experiment("keras_example")

<Experiment: artifact_location='mlflow-artifacts:/470074894443755855', creation_time=1764782216982, experiment_id='470074894443755855', last_update_time=1764782216982, lifecycle_stage='active', name='keras_example', tags={}>

In [5]:
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

In [6]:
# path management
PROJECT_ROOT = '/content/drive/MyDrive/data/Brain Tumor MRI'
PREP_DIR = PROJECT_ROOT + "/data/processed"
ARTEFACTS_DIR = PROJECT_ROOT + "/models"

CLASSES = ["notumor", "glioma", "meningioma", "pituitary"]

# parameters
IMG_SIZE = 260
SEED = 42

## Modeling

### Backbone

In [7]:
# 1. Create DenseNet121 WITHOUT weights
backbone = DenseNet121(
    include_top=False,
    weights=None,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 2. Load RadImageNet weights
backbone.load_weights(ARTEFACTS_DIR + "/RadImageNet-DenseNet121_notop.h5")

# 3. Freeze the backbone for firsts training
backbone.trainable = False

print("✅ RadImageNet DenseNet121 loaded successfully")

✅ RadImageNet DenseNet121 loaded successfully


In [8]:
#backbone.summary()

In [9]:
w = backbone.weights[0].numpy()
print("Mean:", np.mean(w), "Std:", np.std(w))

Mean: -0.0028284893 Std: 0.10494729


Model seems to be correctly loaded.

### Model definition

In [10]:
model_shared_part = keras.Sequential([
    #Data augmentation
    #TODO if need
    # Base
    backbone,
    # Head
    layers.GlobalAveragePooling2D(), # to flatten backbone output but with moderate position importance and more stable for MRI
    layers.Dense(512, use_bias=False), # 512 because it half of the backbone output (1024)
    layers.BatchNormalization(), # to normalize weight before heads
    layers.Activation('relu'),
    #layers.Dropout(0.4) # to reduce over-fitting risks
], name='shared_part')

In [11]:
model_head1 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    #layers.Dropout(0.2),
    layers.Dense(1,activation='sigmoid')
], name='tumor_presence')

In [12]:
model_head2 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    #layers.Dropout(0.2),
    layers.Dense(4,activation='softmax')
], name='tumor_type')

In [13]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = model_shared_part(inputs)
output1 = model_head1(x)
output2 = model_head2(x)

model = keras.Model(
    inputs=inputs,
    outputs={
        "tumor_presence": output1,
        "tumor_type": output2
    },
    name='densenet_two_head'
)

In [14]:
loss_presence = keras.losses.BinaryFocalCrossentropy(
    gamma=2.0,
    alpha=0.25 # to favorize tumor detection, but taking account that tumors are 75% of data
)

In [15]:
def masked_sparse_cce(y_true, y_pred):
    tumor_present = tf.cast(y_true != 0, tf.float32)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    loss = loss * tumor_present
    return tf.reduce_sum(loss) / (tf.reduce_sum(tumor_present) + 1e-6)

In [16]:
model.compile(
    optimizer=keras.optimizers.Adam(), # change learning_rate for 1e-4 in fine-tuning steps
    loss={
        "tumor_presence": loss_presence,
        "tumor_type": masked_sparse_cce,
    },

    metrics={
        "tumor_presence": [
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.AUC(name="auc")
        ],
        "tumor_type": ["accuracy"],
    }
)

In [17]:
#model.summary()

## Streaming Training

In [18]:
def parse_tfrecord(example_proto):
    """
    Parse a single TFRecord example and convert grayscale → RGB.
    """
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }

    example = tf.io.parse_single_example(example_proto, feature_description)

    # Deserialize image
    image = tf.io.parse_tensor(example["image"], out_type=tf.float32)

    # Shape after loading: (260, 260, 3)
    image.set_shape((260, 260, 3))

    label = tf.cast(example["label"], tf.int32)

    return image, label



def load_tfrecord_dataset(tfrecord_dir, shuffle=False, batch_size=1, repeat=False):
    """
    Load a TFRecord dataset from a directory.

    Args:
        tfrecord_dir (str or Path): Folder containing .tfrecord files
        shuffle (bool): Whether to shuffle files and samples
        batch_size (int): Batch size (can stay 1)
        repeat (bool): Repeat dataset indefinitely (for training)

    Returns:
        tf.data.Dataset
    """
    tfrecord_files = tf.io.gfile.glob(
        str(tfrecord_dir) + "/*.tfrecord"
    )

    ds = tf.data.TFRecordDataset(
        tfrecord_files,
        num_parallel_reads=tf.data.AUTOTUNE
    )

    ds = ds.map(
        parse_tfrecord,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        ds = ds.shuffle(buffer_size=512)

    if repeat:
        ds = ds.repeat()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [19]:
TRAIN_DIR = PREP_DIR + "/Training"
VAL_DIR   = PREP_DIR + "/Validation"
TEST_DIR  = PREP_DIR + "/Testing"

train_ds = load_tfrecord_dataset(
    TRAIN_DIR,
    shuffle=True,
    batch_size=1,
    repeat=False
)

val_ds = load_tfrecord_dataset(
    VAL_DIR,
    shuffle=False,
    batch_size=1,
    repeat=False
)
"""
test_ds = load_tfrecord_dataset(
    TEST_DIR,
    shuffle=False,
    batch_size=1,
    repeat=False
)
"""

'\ntest_ds = load_tfrecord_dataset(\n    TEST_DIR,\n    shuffle=False,\n    batch_size=1,\n    repeat=False\n)\n'

In [20]:
def split_labels(image, label):
    """
    Create labels for a 2-head model.
    """
    tumor_present = tf.cast(label != 0, tf.float32)
    tumor_type = tf.cast(label, tf.int32)

    return image, {
        "tumor_presence": tumor_present,
        "tumor_type": tumor_type
    }


In [21]:
train_ds = train_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)

In [22]:
for x, y in train_ds.take(1):
    print("Image:")
    print(x.dtype, x.shape)
    print("\nLabels:")
    for k, v in y.items():
        print(k, v.dtype, v.shape)

Image:
<dtype: 'float32'> (1, 260, 260, 3)

Labels:
tumor_presence <dtype: 'float32'> (1,)
tumor_type <dtype: 'int32'> (1,)


In [23]:
def count_tfrecord_samples(directory_path):
    """
    Count the number of TFRecord files in a directory.
    Assumes 1 sample per TFRecord.
    """
    directory_path = Path(directory_path)
    return len(list(directory_path.glob("*.tfrecord")))

In [24]:
def debugg_ds_size(train_ds, val_ds):
  print(">>>")
  print("Train cardinality:", tf.data.experimental.cardinality(train_ds).numpy())
  print("Val cardinality:", tf.data.experimental.cardinality(val_ds).numpy())

  print("Testing train_ds iteration")
  for batch in train_ds.take(1):
      print("Train batch OK")
      break
  else:
      print("❌ train_ds is EMPTY")

  print("Testing val_ds iteration")
  for batch in val_ds.take(1):
      print("Val batch OK")
      break
  else:
      print("❌ val_ds is EMPTY")

  print("<<<\n")

In [25]:
print(train_ds.element_spec)

(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), {'tumor_presence': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'tumor_type': TensorSpec(shape=(None,), dtype=tf.int32, name=None)})


In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_tumor_presence_recall",
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_tumor_presence_recall",
    mode="max",
    min_delta=0.001,
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    #steps_per_epoch=count_tfrecord_samples(TRAIN_DIR),
    #validation_steps=count_tfrecord_samples(VAL_DIR),
    callbacks=[reduce_lr, early_stopping],
)

Epoch 1/50
    687/Unknown 182s 230ms/step - loss: 1.1164 - tumor_presence_accuracy: 0.6929 - tumor_presence_auc: 0.4854 - tumor_presence_loss: 0.1672 - tumor_presence_precision: 0.6987 - tumor_presence_recall: 0.9858 - tumor_type_accuracy: 0.2505 - tumor_type_loss: 0.9492

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot();
print("Minimum validation loss: {}".format(history_df['val_loss'].min()))

In [ ]:
Warning : do not forget :
- integerer mlflow -> premier essai avec visu
- activer dropoutS dans model
- stratégie de fine-tuning du backbone
- BatchNormalization precaution in fine-tuning
- maximize reccal (y_pred > 0.3 → tumeur)